# 第二章 Transformer 架构：源码汇总与维度可视化

本 Notebook 汇总 [第二章 Transformer架构.md](./第二章%20Transformer架构.md) 中出现的全部示例源码，并保留原注释，按正文顺序组织：

- 2.1 注意力机制：注意力计算、自注意力、掩码自注意力、多头注意力
- 2.2 Encoder-Decoder：前馈神经网络、层归一化、残差连接、Encoder、Decoder
- 2.3 搭建一个 Transformer：Embedding、位置编码、完整 Transformer

> 为观察各部分的数据输入输出与维度变化，各组件在关键步骤加入了 `show()` 打印，演示统一使用：
> `batch_size = 2`、`seq_len = 8`、`n_embd = dim = 32`、`n_heads = 4`、`head_dim = 8`、`vocab_size = 100`。
>
> 正文中个别代码片段依赖 `self`（位于 `forward` 方法内部），无法单独运行，已用 Markdown 代码块原样列出；其余代码均可在 Notebook 中直接执行。

## 0. 环境准备

运行本 Notebook 需要 `torch` 与 `numpy`。为使正文代码可直接运行，这里补充了 `ModelArgs` 配置类（与 `code/transformer.py` 一致）、一个演示用的小型参数实例，以及用于打印张量维度的 `show()` 函数。

In [39]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass


@dataclass
class ModelArgs:
    n_embd: int  # 嵌入维度
    n_heads: int  # 头数
    dim: int  # 模型维度
    dropout: float
    max_seq_len: int
    vocab_size: int
    block_size: int
    n_layer: int


# 演示用的小型配置（原书示例参数见 code/transformer.py 的 main 函数）
args = ModelArgs(
    n_embd=32,
    n_heads=4,
    dim=32,
    dropout=0.1,
    max_seq_len=64,
    vocab_size=100,
    block_size=64,
    n_layer=2,
)


def show(name, tensor):
    '''打印张量名称与形状，便于观察各组件的数据维度变化'''
    print(f"{name}: {tuple(tensor.shape)}")


## 2.1 注意力机制

### 2.1.3 注意力机制的实现

注意力机制的核心公式为：

$$
\text{attention}(Q,K,V)=\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

以下实现假设输入的 `query`、`key`、`value` 是已经过参数矩阵转化的词向量矩阵，即公式中的 Q、K、V。代码中打印了 `(B, T, d_k)` → `(B, T, T)` → `(B, T, d_v)` 的维度变化。

In [40]:
'''注意力计算函数'''
def attention(query, key, value, dropout=None):
    '''
    args:
    query: 查询值矩阵
    key: 键值矩阵
    value: 真值矩阵
    '''
    # 获取键向量的维度，键向量的维度和值向量的维度相同
    d_k = query.size(-1)
    show("attention 输入 query", query)
    show("attention 输入 key", key)
    show("attention 输入 value", value)
    print(f"attention d_k: {d_k}")
    # 计算Q与K的内积并除以根号dk
    # transpose——相当于转置 key.transpose(-2, -1)将key的后两维交换位置，本例中转置前为(2, 8, 16)，转置后为(2, 16, 8)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    show("QK^T / sqrt(d_k) -> scores", scores)
    # Softmax
    p_attn = scores.softmax(dim=-1)
    show("softmax -> p_attn", p_attn)
    # 随机“丢弃”一部分注意力权重，防止过拟合。主要进行：1、以概率 p 随机把一部分元素置为 0；2、把剩下的元素放大 1/(1-p) 倍（保证期望值不变）
    if dropout is not None:
        p_attn = dropout(p_attn)
        # 采样
    # 根据计算结果对value进行加权求和
    output = torch.matmul(p_attn, value)
    show("p_attn @ value -> output", output)
    return output, p_attn


In [41]:
# 演示：batch_size=2, seq_len=8, hidden_size=16
query = torch.randn(2, 8, 16)
key = torch.randn(2, 8, 16)
value = torch.randn(2, 8, 16)
output, p_attn = attention(query, key, value)
show("注意力最终输出", output)
show("注意力权重矩阵", p_attn)


attention 输入 query: (2, 8, 16)
attention 输入 key: (2, 8, 16)
attention 输入 value: (2, 8, 16)
attention d_k: 16
QK^T / sqrt(d_k) -> scores: (2, 8, 8)
softmax -> p_attn: (2, 8, 8)
p_attn @ value -> output: (2, 8, 16)
注意力最终输出: (2, 8, 16)
注意力权重矩阵: (2, 8, 8)


### 2.1.4 自注意力

自注意力中 Q、K、V 由同一个输入经过不同的参数矩阵得到，代码上表现为给 `attention` 传入同一个输入：输入 `(B, T, d)` 经过注意力后输出仍为 `(B, T, d)`，同时得到 `(B, T, T)` 的注意力权重。

In [19]:
# attention 为上文定义的注意力计算函数
x = torch.randn(2, 8, 16)  # 演示输入：(batch_size, seq_len, hidden_size)
show("自注意力输入 x", x)
self_attn_out, self_attn_weight = attention(x, x, x)
show("自注意力输出", self_attn_out)
show("自注意力权重", self_attn_weight)


自注意力输入 x: (2, 8, 16)
attention 输入 query: (2, 8, 16)
attention 输入 key: (2, 8, 16)
attention 输入 value: (2, 8, 16)
attention d_k: 16
QK^T / sqrt(d_k) -> scores: (2, 8, 8)
softmax -> p_attn: (2, 8, 8)
p_attn @ value -> output: (2, 8, 16)
自注意力输出: (2, 8, 16)
自注意力权重: (2, 8, 8)


### 2.1.5 掩码自注意力

掩码自注意力通过上三角掩码遮蔽未来信息：上三角位置置为 `-inf`，经过 Softmax 后这些位置的权重变为 0。下面打印掩码矩阵的形状、位置以及加掩码前后的注意力分数。

In [20]:
# 创建一个上三角矩阵，用于遮蔽未来信息。
# 先通过 full 函数创建一个 1 * seq_len * seq_len 的矩阵
mask = torch.full((1, args.max_seq_len, args.max_seq_len), float("-inf"))
# triu 函数的功能是创建一个上三角矩阵
mask = torch.triu(mask, diagonal=1)

show("mask", mask)
print("被遮蔽（-inf）的位置数量:", int(torch.isinf(mask).sum()))
print("mask 左上角 4x4 预览:")
print(mask[0, :4, :4])


mask: (1, 64, 64)
被遮蔽（-inf）的位置数量: 2016
mask 左上角 4x4 预览:
tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])


In [21]:
# 为便于运行，先构造演示用的注意力分数与输入
seqlen = 4
xq = torch.randn(1, 2, seqlen, 8)
scores = torch.randn(1, 2, seqlen, seqlen)
show("原始 scores", scores)

# 此处的 scores 为计算得到的注意力分数，mask 为上文生成的掩码矩阵
scores = scores + mask[:, :seqlen, :seqlen]
show("加上掩码后的 scores", scores)
scores = F.softmax(scores.float(), dim=-1).type_as(xq)
show("softmax 后的 scores", scores)
print("第 1 个样本、第 1 个头的注意力权重（上三角已被置 0）:")
print(scores[0, 0])


原始 scores: (1, 2, 4, 4)
加上掩码后的 scores: (1, 2, 4, 4)
softmax 后的 scores: (1, 2, 4, 4)
第 1 个样本、第 1 个头的注意力权重（上三角已被置 0）:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.7001, 0.2999, 0.0000, 0.0000],
        [0.3269, 0.4779, 0.1952, 0.0000],
        [0.8050, 0.0247, 0.0627, 0.1075]])


### 2.1.6 多头注意力

多头注意力将输入拆分为 `n_heads` 组分别做注意力计算，再将结果拼接并经过输出投影。维度变化为：

`(B, T, n_embd)` → 线性投影 `(B, T, dim)` → 拆头 `(B, n_heads, T, head_dim)` → scores `(B, n_heads, T, T)` → 加权求和 `(B, n_heads, T, head_dim)` → 合并 `(B, T, dim)`。

> 注意：原书示例默认 Q、K、V 的序列长度相同；为支持 Decoder 交叉注意力（Query 与 Key/Value 长度不同），下面代码在 `view` 与掩码截取时分别使用各自的序列长度。

In [22]:
'''多头自注意力计算模块'''
class MultiHeadAttention(nn.Module):

    def __init__(self, args: ModelArgs, is_causal=False):
        # 构造函数
        # args: 配置对象
        super().__init__()
        # 隐藏层维度必须是头数的整数倍，因为后面我们会将输入拆成头数个矩阵
        assert args.dim % args.n_heads == 0
        # 每个头的维度，等于模型维度除以头的总数。
        self.head_dim = args.dim // args.n_heads
        self.n_heads = args.n_heads

        # Wq, Wk, Wv 参数矩阵，每个参数矩阵为 n_embd x dim
        # 这里通过三个组合矩阵来代替了n个参数矩阵的组合，其逻辑在于矩阵内积再拼接其实等同于拼接矩阵再内积，
        # 不理解的读者可以自行模拟一下，每一个线性层其实相当于n个参数矩阵的拼接
        # nn.Linear(in_features, out_features, bias=True) 创建全连接层
        self.wq = nn.Linear(args.n_embd, self.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.n_embd, self.n_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.n_embd, self.n_heads * self.head_dim, bias=False)
        # 输出权重矩阵，维度为 dim x dim（head_dim = dim / n_heads）
        self.wo = nn.Linear(self.n_heads * self.head_dim, args.dim, bias=False)
        # 注意力的 dropout
        self.attn_dropout = nn.Dropout(args.dropout)
        # 残差连接的 dropout
        self.resid_dropout = nn.Dropout(args.dropout)
        self.is_causal = is_causal

        # 创建一个上三角矩阵，用于遮蔽未来信息
        # 注意，因为是多头注意力，Mask 矩阵比之前我们定义的多一个维度
        if is_causal:
            mask = torch.full((1, 1, args.max_seq_len, args.max_seq_len), float("-inf"))
            mask = torch.triu(mask, diagonal=1)
            # 注册为模型的缓冲区
            self.register_buffer("mask", mask)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):

        # 获取批次大小和序列长度，[batch_size, seq_len, dim]
        # Decoder 的交叉注意力中，Query 与 Key/Value 的序列长度可能不同，因此分别获取
        bsz, seqlen_q, _ = q.shape
        _, seqlen_k, _ = k.shape
        _, seqlen_v, _ = v.shape
        show("MultiHeadAttention 输入 q", q)
        show("MultiHeadAttention 输入 k", k)
        show("MultiHeadAttention 输入 v", v)

        # 计算查询（Q）、键（K）、值（V）,输入通过参数矩阵层，维度为 (B, T, n_embed) x (n_embed, dim) -> (B, T, dim)
        xq, xk, xv = self.wq(q), self.wk(k), self.wv(v)
        show("线性投影后 xq", xq)
        show("线性投影后 xk", xk)
        show("线性投影后 xv", xv)

        # 将 Q、K、V 拆分成多头，维度为 (B, T, n_head, dim // n_head)，然后交换维度，变成 (B, n_head, T, dim // n_head)
        # 因为在注意力计算中我们是取了后两个维度参与计算
        # 为什么要先按B*T*n_head*C//n_head展开再互换1、2维度而不是直接按注意力输入展开，是因为view的展开方式是直接把输入全部排开，
        # 然后按要求构造，可以发现只有上述操作能够实现我们将每个头对应部分取出来的目标
        xq = xq.view(bsz, seqlen_q, self.n_heads, self.head_dim)
        xk = xk.view(bsz, seqlen_k, self.n_heads, self.head_dim)
        xv = xv.view(bsz, seqlen_v, self.n_heads, self.head_dim)
        show("拆分为多头后 xq（view）", xq)
        xq = xq.transpose(1, 2)
        xk = xk.transpose(1, 2)
        xv = xv.transpose(1, 2)
        show("交换维度后 xq（transpose）", xq)
        print(f"头数 n_heads: {self.n_heads}, 每个头维度 head_dim: {self.head_dim}")

        # 注意力计算
        # 计算 QK^T / sqrt(d_k)，维度为 (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        scores = torch.matmul(xq, xk.transpose(2, 3)) / math.sqrt(self.head_dim)
        show("QK^T / sqrt(d_k) -> scores", scores)
        # 掩码自注意力必须有注意力掩码
        if self.is_causal:
            assert hasattr(self, 'mask')
            # 这里截取到序列长度，因为有些序列可能比 max_seq_len 短
            scores = scores + self.mask[:, :, :seqlen_q, :seqlen_k]
            show("加上因果掩码后 scores", scores)
        # 计算 softmax，维度为 (B, nh, T, T)
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)
        show("softmax 后 scores", scores)
        # 做 Dropout
        scores = self.attn_dropout(scores)
        # V * Score，维度为(B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        output = torch.matmul(scores, xv)
        show("scores @ xv -> output", output)

        # 恢复时间维度并合并头。
        # 将多头的结果拼接起来, 先交换维度为 (B, T, n_head, dim // n_head)，再拼接成 (B, T, n_head * dim // n_head)
        # contiguous 函数用于重新开辟一块新内存存储，因为Pytorch设置先transpose再view会报错，
        # 因为view直接基于底层存储得到，然而transpose并不会改变底层存储，因此需要额外存储
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen_q, -1)
        show("合并多头后 output", output)

        # 最终投影回残差流。
        output = self.wo(output)
        output = self.resid_dropout(output)
        show("输出投影后 output", output)
        return output


In [23]:
# 演示：多头自注意力（非掩码）
mha = MultiHeadAttention(args)
src = torch.randn(2, 8, args.n_embd)
show("多头注意力演示输入 src", src)
mha_out = mha(src, src, src)
show("多头注意力最终输出", mha_out)


多头注意力演示输入 src: (2, 8, 32)
MultiHeadAttention 输入 q: (2, 8, 32)
MultiHeadAttention 输入 k: (2, 8, 32)
MultiHeadAttention 输入 v: (2, 8, 32)
线性投影后 xq: (2, 8, 32)
线性投影后 xk: (2, 8, 32)
线性投影后 xv: (2, 8, 32)
拆分为多头后 xq（view）: (2, 8, 4, 8)
交换维度后 xq（transpose）: (2, 4, 8, 8)
头数 n_heads: 4, 每个头维度 head_dim: 8
QK^T / sqrt(d_k) -> scores: (2, 4, 8, 8)
softmax 后 scores: (2, 4, 8, 8)
scores @ xv -> output: (2, 4, 8, 8)
合并多头后 output: (2, 8, 32)
输出投影后 output: (2, 8, 32)
多头注意力最终输出: (2, 8, 32)


## 2.2 Encoder-Decoder

### 2.2.2 前馈神经网络

前馈神经网络由两个线性层中间加一个 ReLU 激活函数组成，并加入 Dropout 防止过拟合。在 Transformer 中两层维度相同，因此输入输出维度均为 `(B, T, dim)`。

In [24]:
class MLP(nn.Module):
    '''前馈神经网络'''
    def __init__(self, dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        # 定义第一层线性变换，从输入维度到隐藏维度
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        # 定义第二层线性变换，从隐藏维度到输入维度
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        # 定义dropout层，用于防止过拟合
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # 前向传播函数
        show("MLP 输入 x", x)
        # 首先，输入x通过第一层线性变换和RELU激活函数
        h = self.w1(x)
        show("w1 线性变换后", h)
        h = F.relu(h)
        show("ReLU 激活后", h)
        # 最后，通过第二层线性变换和dropout层
        out = self.dropout(self.w2(h))
        show("w2 + dropout 后", out)
        return out


In [25]:
# 演示：隐藏维度扩展到 4 倍，可清楚看到先升维再降维
mlp = MLP(args.dim, args.dim * 4, args.dropout)
x = torch.randn(2, 8, args.dim)
show("MLP 演示输入", x)
mlp_out = mlp(x)
show("MLP 最终输出", mlp_out)


MLP 演示输入: (2, 8, 32)
MLP 输入 x: (2, 8, 32)
w1 线性变换后: (2, 8, 128)
ReLU 激活后: (2, 8, 128)
w2 + dropout 后: (2, 8, 32)
MLP 最终输出: (2, 8, 32)


### 2.2.3 层归一化

Layer Norm 在每个样本的所有维度上计算均值与方差，使每个样本的分布达到稳定。实现中包含两个可学习的线性映射参数 `a_2`、`b_2`，输入输出维度保持不变。

In [26]:
class LayerNorm(nn.Module):
    ''' Layer Norm 层'''
    def __init__(self, features, eps=1e-6):
        super().__init__()
        # 线性矩阵做映射
        # self.a_2是LayerNorm 的缩放系数（scale / γ），也叫增益参数。
        self.a_2 = nn.Parameter(torch.ones(features))
        # self.b_2是LayerNorm 的偏移系数（shift / β），也叫偏置参数。
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        # 在统计每个样本所有维度的值，求均值和方差
        mean = x.mean(-1, keepdim=True) # mean: [bsz, max_len, 1]
        std = x.std(-1, keepdim=True) # std: [bsz, max_len, 1]
        show("LayerNorm 输入 x", x)
        show("按最后一维求均值 mean", mean)
        show("按最后一维求标准差 std", std)
        # 注意这里也在最后一个维度发生了广播
        # out = self.a_2 * (x - mean) / (std + self.eps) + self.b_2 如果没有self.a_2、self.b_2，输出均值必然为 0，方差必然为 1。
        # 问题在于：1、强制标准化会破坏网络的表达能力：网络可能需要某些特征保留原来的分布（比如非零均值）才能更好地学习。
        # 2、有了 γ 和 β，网络可以“自己决定”要不要标准化、标准化到什么程度：如果学到的结果是 γ=1, β=0，就等价于完全归一化；如果学到其他值，就能恢复或调整分布。
        out = self.a_2 * (x - mean) / (std + self.eps) + self.b_2
        show("LayerNorm 输出 out", out)
        return out


In [27]:
# 演示：LayerNorm 不改变维度
ln = LayerNorm(args.n_embd)
x = torch.randn(2, 8, args.n_embd)
show("LayerNorm 演示输入", x)
ln_out = ln(x)
show("LayerNorm 最终输出", ln_out)


LayerNorm 演示输入: (2, 8, 32)
LayerNorm 输入 x: (2, 8, 32)
按最后一维求均值 mean: (2, 8, 1)
按最后一维求标准差 std: (2, 8, 1)
LayerNorm 输出 out: (2, 8, 32)
LayerNorm 最终输出: (2, 8, 32)


### 2.2.4 残差连接

残差连接让下一层的输入同时包含上一层的输出与输入：

$$
x = x + \mathrm{MultiHeadSelfAttention}(\mathrm{LayerNorm}(x))
$$

$$
output = x + \mathrm{FNN}(\mathrm{LayerNorm}(x))
$$

在代码中，残差连接体现为在 `forward` 中把子层输出与原输入相加（摘自 `EncoderLayer.forward`，注释中标出了维度）：

```python
# 注意力计算：x (B, T, n_embd) + 注意力输出 (B, T, dim) -> h (B, T, dim)
# self.attention_norm(x)	先对输入 x 做 LayerNorm，稳定数值分布
# self.attention.forward(...)	归一化后的结果送入多头注意力模块，让 token 之间交互信息（如 Self-Attention 计算 Q、K、V）
# x + (...)	残差连接：把注意力的输出加回原始输入 x
h = x + self.attention.forward(self.attention_norm(x))
# 经过前馈神经网络：h (B, T, dim) -> out (B, T, dim)
out = h + self.feed_forward.forward(self.fnn_norm(h))
```

上文中 `self.attention_norm` 和 `self.fnn_norm` 都是 LayerNorm 层，`self.attention` 是注意力层，`self.feed_forward` 是前馈神经网络。

### 2.2.5 Encoder

Encoder 由 N 个 Encoder Layer 组成，每个 Layer 包含一个多头自注意力层和一个前馈神经网络，最后接一个 Layer Norm。整个 Encoder 的输入输出维度均为 `(B, T, n_embd)`。

In [28]:
class EncoderLayer(nn.Module):
    '''Encoder层'''
    def __init__(self, args):
        super().__init__()
        # 一个 Layer 中有两个 LayerNorm，分别在 Attention 之前和 MLP 之前
        self.attention_norm = LayerNorm(args.n_embd)
        # Encoder 不需要掩码，传入 is_causal=False
        self.attention = MultiHeadAttention(args, is_causal=False)
        self.fnn_norm = LayerNorm(args.n_embd)
        self.feed_forward = MLP(args.dim, args.dim, args.dropout)

    def forward(self, x):
        show("EncoderLayer 输入 x", x)
        # Layer Norm
        norm_x = self.attention_norm(x)
        # 自注意力
        h = x + self.attention.forward(norm_x, norm_x, norm_x)
        show("掩码/自注意力 + 残差后 h", h)
        # 经过前馈神经网络
        out = h + self.feed_forward.forward(self.fnn_norm(h))
        show("前馈网络 + 残差后 out", out)
        return out


class Encoder(nn.Module):
    '''Encoder 块'''
    def __init__(self, args):
        super(Encoder, self).__init__()
        # 一个 Encoder 由 N 个 Encoder Layer 组成
        self.layers = nn.ModuleList([EncoderLayer(args) for _ in range(args.n_layer)])
        self.norm = LayerNorm(args.n_embd)

    def forward(self, x):
        "分别通过 N 层 Encoder Layer"
        for i, layer in enumerate(self.layers):
            print(f"--- Encoder Layer {i} ---")
            x = layer(x)
        x = self.norm(x)
        show("Encoder 最终输出", x)
        return x


In [29]:
# 演示：Encoder 的输入输出维度一致
encoder = Encoder(args)
enc_input = torch.randn(2, 8, args.n_embd)
show("Encoder 演示输入", enc_input)
enc_out = encoder(enc_input)
show("Encoder 演示输出 enc_out", enc_out)


Encoder 演示输入: (2, 8, 32)
--- Encoder Layer 0 ---
EncoderLayer 输入 x: (2, 8, 32)
LayerNorm 输入 x: (2, 8, 32)
按最后一维求均值 mean: (2, 8, 1)
按最后一维求标准差 std: (2, 8, 1)
LayerNorm 输出 out: (2, 8, 32)
MultiHeadAttention 输入 q: (2, 8, 32)
MultiHeadAttention 输入 k: (2, 8, 32)
MultiHeadAttention 输入 v: (2, 8, 32)
线性投影后 xq: (2, 8, 32)
线性投影后 xk: (2, 8, 32)
线性投影后 xv: (2, 8, 32)
拆分为多头后 xq（view）: (2, 8, 4, 8)
交换维度后 xq（transpose）: (2, 4, 8, 8)
头数 n_heads: 4, 每个头维度 head_dim: 8
QK^T / sqrt(d_k) -> scores: (2, 4, 8, 8)
softmax 后 scores: (2, 4, 8, 8)
scores @ xv -> output: (2, 4, 8, 8)
合并多头后 output: (2, 8, 32)
输出投影后 output: (2, 8, 32)
掩码/自注意力 + 残差后 h: (2, 8, 32)
LayerNorm 输入 x: (2, 8, 32)
按最后一维求均值 mean: (2, 8, 1)
按最后一维求标准差 std: (2, 8, 1)
LayerNorm 输出 out: (2, 8, 32)
MLP 输入 x: (2, 8, 32)
w1 线性变换后: (2, 8, 32)
ReLU 激活后: (2, 8, 32)
w2 + dropout 后: (2, 8, 32)
前馈网络 + 残差后 out: (2, 8, 32)
--- Encoder Layer 1 ---
EncoderLayer 输入 x: (2, 8, 32)
LayerNorm 输入 x: (2, 8, 32)
按最后一维求均值 mean: (2, 8, 1)
按最后一维求标准差 std: (2, 8, 1)
LayerNo

### 2.2.6 Decoder

Decoder Layer 由三个子层组成：掩码自注意力层、以 Encoder 输出为 K/V 的多头注意力层、前馈神经网络。交叉注意力中 Query 来自 Decoder（长度 `T_tgt`），Key/Value 来自 Encoder 输出（长度 `T_src`），因此注意力分数为 `(B, n_heads, T_tgt, T_src)`。

In [30]:
class DecoderLayer(nn.Module):
    '''解码层'''
    def __init__(self, args):
        super().__init__()
        # 一个 Layer 中有三个 LayerNorm，分别在 Mask Attention 之前、Self Attention 之前和 MLP 之前
        self.attention_norm_1 = LayerNorm(args.n_embd)
        # Decoder 的第一个部分是 Mask Attention，传入 is_causal=True
        self.mask_attention = MultiHeadAttention(args, is_causal=True)
        self.attention_norm_2 = LayerNorm(args.n_embd)
        # Decoder 的第二个部分是 类似于 Encoder 的 Attention，传入 is_causal=False
        self.attention = MultiHeadAttention(args, is_causal=False)
        self.ffn_norm = LayerNorm(args.n_embd)
        # 第三个部分是 MLP
        self.feed_forward = MLP(args.dim, args.dim, args.dropout)

    def forward(self, x, enc_out):
        show("DecoderLayer 输入 x", x)
        show("DecoderLayer 输入 enc_out", enc_out)
        # Layer Norm
        norm_x = self.attention_norm_1(x)
        # 掩码自注意力
        x = x + self.mask_attention.forward(norm_x, norm_x, norm_x)
        show("掩码自注意力 + 残差后 x", x)
        # 多头注意力
        norm_x = self.attention_norm_2(x)
        h = x + self.attention.forward(norm_x, enc_out, enc_out)
        show("交叉注意力 + 残差后 h", h)
        # 经过前馈神经网络
        out = h + self.feed_forward.forward(self.ffn_norm(h))
        show("前馈网络 + 残差后 out", out)
        return out


class Decoder(nn.Module):
    '''解码器'''
    def __init__(self, args):
        super(Decoder, self).__init__()
        # 一个 Decoder 由 N 个 Decoder Layer 组成
        self.layers = nn.ModuleList([DecoderLayer(args) for _ in range(args.n_layer)])
        self.norm = LayerNorm(args.n_embd)

    def forward(self, x, enc_out):
        "Pass the input (and mask) through each layer in turn."
        for i, layer in enumerate(self.layers):
            print(f"--- Decoder Layer {i} ---")
            x = layer(x, enc_out)
        x = self.norm(x)
        show("Decoder 最终输出", x)
        return x


In [31]:
# 演示：Decoder 输入长度 6、Encoder 输出长度 8，可观察交叉注意力的维度变化
decoder = Decoder(args)
dec_input = torch.randn(2, 6, args.n_embd)
show("Decoder 演示输入 dec_input", dec_input)
dec_out = decoder(dec_input, enc_out)
show("Decoder 演示输出 dec_out", dec_out)


Decoder 演示输入 dec_input: (2, 6, 32)
--- Decoder Layer 0 ---
DecoderLayer 输入 x: (2, 6, 32)
DecoderLayer 输入 enc_out: (2, 8, 32)
LayerNorm 输入 x: (2, 6, 32)
按最后一维求均值 mean: (2, 6, 1)
按最后一维求标准差 std: (2, 6, 1)
LayerNorm 输出 out: (2, 6, 32)
MultiHeadAttention 输入 q: (2, 6, 32)
MultiHeadAttention 输入 k: (2, 6, 32)
MultiHeadAttention 输入 v: (2, 6, 32)
线性投影后 xq: (2, 6, 32)
线性投影后 xk: (2, 6, 32)
线性投影后 xv: (2, 6, 32)
拆分为多头后 xq（view）: (2, 6, 4, 8)
交换维度后 xq（transpose）: (2, 4, 6, 8)
头数 n_heads: 4, 每个头维度 head_dim: 8
QK^T / sqrt(d_k) -> scores: (2, 4, 6, 6)
加上因果掩码后 scores: (2, 4, 6, 6)
softmax 后 scores: (2, 4, 6, 6)
scores @ xv -> output: (2, 4, 6, 8)
合并多头后 output: (2, 6, 32)
输出投影后 output: (2, 6, 32)
掩码自注意力 + 残差后 x: (2, 6, 32)
LayerNorm 输入 x: (2, 6, 32)
按最后一维求均值 mean: (2, 6, 1)
按最后一维求标准差 std: (2, 6, 1)
LayerNorm 输出 out: (2, 6, 32)
MultiHeadAttention 输入 q: (2, 6, 32)
MultiHeadAttention 输入 k: (2, 8, 32)
MultiHeadAttention 输入 v: (2, 8, 32)
线性投影后 xq: (2, 6, 32)
线性投影后 xk: (2, 8, 32)
线性投影后 xv: (2, 8, 32)
拆分为多头后 xq（

## 2.3 搭建一个 Transformer

### 2.3.1 Embedding 层

Embedding 层是存储固定大小词典的嵌入向量查找表，输入为 token 经 tokenizer 转化后的 index，输出为 `(batch_size, seq_len, embedding_dim)` 的词向量矩阵。Transformer 中直接使用 `nn.Embedding`：

```python
self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)
```

下面演示 token index 到词向量的维度变化：

In [32]:
# 演示：batch_size=1, seq_len=3 的 token index
input_ids = torch.tensor([[0, 1, 2]])
show("token index 输入", input_ids)
tok_embeddings = nn.Embedding(args.vocab_size, args.dim)
token_vec = tok_embeddings(input_ids)
show("Embedding 输出", token_vec)


token index 输入: (1, 3)
Embedding 输出: (1, 3, 32)


### 2.3.2 位置编码

位置编码使用正余弦函数：

$$
PE(pos, 2i) = \sin(pos/10000^{2i/d_{model}}),\qquad
PE(pos, 2i+1) = \cos(pos/10000^{2i/d_{model}})
$$

pos：token 在句子中的位置（0, 1, 2, 3, …）
2i / 2i+1：维度索引（偶数维用 sin，奇数维用 cos）
d_model：词向量维度，本例中为 4
sin 和 cos 总是成对出现，共享同一个 2i。第 0 维和第 1 维是一对（2i=0），第 2 维和第 3 维是一对（2i=2）。

以下为正文中用于演示位置编码数值的示例代码（原示例中导入的 `matplotlib` 并未使用，此处省略以保证可直接运行），输出的矩阵形状为 `(seq_len, d_model)`：

In [33]:
def PositionEncoding(seq_len, d_model, n=10000):
    P = np.zeros((seq_len, d_model))
    for k in range(seq_len):
        for i in np.arange(int(d_model/2)):
            denominator = np.power(n, 2*i/d_model)
            P[k, 2*i] = np.sin(k/denominator)
            P[k, 2*i+1] = np.cos(k/denominator)
    return P

P = PositionEncoding(seq_len=4, d_model=4, n=100)
show("位置编码矩阵 P", P)
print(P)


位置编码矩阵 P: (4, 4)
[[ 0.          1.          0.          1.        ]
 [ 0.84147098  0.54030231  0.09983342  0.99500417]
 [ 0.90929743 -0.41614684  0.19866933  0.98006658]
 [ 0.14112001 -0.9899925   0.29552021  0.95533649]]


基于上述原理，实现位置编码层。该层将形状为 `(1, block_size, n_embd)` 的位置编码加到输入 `(B, T, n_embd)` 上，输出维度不变。

## `div_term` 详解：计算公式中的分母项

### 先看数学目标

位置编码公式中，分母是：

$$10000^{2i/d_{model}}$$

但代码中**不直接计算幂**，而是用指数形式。根据恒等式 $a^x = e^{x \ln a}$：

$$10000^{2i/d_{model}} = e^{\frac{2i}{d_{model}} \cdot \ln 10000} = e^{2i \cdot \frac{\ln 10000}{d_{model}}}$$

代码先算出**指数部分**，再用 `torch.exp` 还原，两者完全等价。

---

### 逐段拆解

#### ① `torch.arange(0, args.n_embd, 2)`

生成偶数序列：0, 2, 4, 6, ...（步长为 2）

```python
# 假设 n_embd = 6
tensor([0, 2, 4])   # 即公式中的 2i，i = 0, 1, 2
```

这正好对应公式中的 **2i**——只有偶数维需要算频率，奇数维（cos）与它前面的偶数维**共用**同一个频率。

#### ② `-(math.log(10000.0) / args.n_embd)`

计算标量 $-\dfrac{\ln 10000}{d_{model}}$

- `math.log(10000.0)` = $\ln 10000 \approx 9.21$
- 除以 `n_embd`（即 $d_{model}$）
- 加负号：因为要放在指数上做**除法**（$e^{-x} = 1/e^x$）

#### ③ 两者相乘

$$2i \times \left(-\frac{\ln 10000}{d_{model}}\right) = -\frac{2i \cdot \ln 10000}{d_{model}}$$

```python
# n_embd = 6 时
tensor([0, 2, 4]) * -(9.21/6)
= tensor([0, -3.07, -6.14])
```

#### ④ `torch.exp(...)`

对指数取 e 的幂，还原出真正的频率分母：

```python
torch.exp([0, -3.07, -6.14])
= tensor([1, 0.0464, 0.00215])
```

验证：$10000^{0}=1$，$10000^{2/6}\approx 0.0464$，$10000^{4/6}\approx 0.00215$ ✓

---

### 为什么绕这么大一圈，不直接写 `10000 ** (2i/n_embd)`？

**功能上完全等价**，直接写幂也行：

```python
div_term = 10000 ** (torch.arange(0, n_embd, 2) / n_embd)  # 等价写法
```

用 `exp` 写法的实际原因：

1. **数值稳定性**：当 2i 很大时，$10000^{2i/d}$ 会变成极大的数，先算负指数再取 exp 中间值都较小，在某些底层实现中更稳定
2. **习惯/惯例**：这种写法来自原论文官方实现及 OpenAI GPT 代码，后续实现纷纷沿用
3. **效率差别可忽略**：现代硬件上两种写法性能几乎一样

---

### div_term 在后续的作用

```python
pe[:, 0::2] = torch.sin(position * div_term)  # 偶数维: sin(pos / div_term)
pe[:, 1::2] = torch.cos(position * div_term)  # 奇数维: cos(pos / div_term)
```

`div_term` 就是公式里的 $10000^{2i/d_{model}}$，形状 `(n_embd/2,)`，与列向量 `position` 广播相乘后得到 $pos/10000^{2i/d_{model}}$。

**它的物理意义**：产生一组从 1 递减到约 $1/10000$ 的频率——

- **前面的维度**（div_term ≈ 1）：正弦波波长短、变化快 → 编码**精细的局部位置**
- **后面的维度**（div_term ≈ 0.0001）：波长极长、变化慢 → 编码**粗粒度的全局位置**

多尺度的波长组合让每个位置都获得一个独一无二的“指纹”，这就是位置编码能区分不同位置的根本原因。

### 总结对照表

| 代码片段 | 对应数学 |
|---------|---------|
| `torch.arange(0, n_embd, 2)` | $2i = 0, 2, 4, \dots$ |
| `math.log(10000.0) / n_embd` | $\ln 10000 / d_{model}$ |
| `* -(负号)` | 转成负指数实现除法 |
| `torch.exp(...)` | $e^{-x \ln 10000} = 10000^{-x}$，还原出 $10000^{2i/d_{model}}$ |


In [34]:
class PositionalEncoding(nn.Module):
    '''位置编码模块'''

    def __init__(self, args):
        super(PositionalEncoding, self).__init__()
        # Dropout 层
        # self.dropout = nn.Dropout(p=args.dropout)

        # block size 是序列的最大长度
        pe = torch.zeros(args.block_size, args.n_embd)
        # unsqueeze(1)将向量变为列向量
        position = torch.arange(0, args.block_size).unsqueeze(1)
        # 计算 theta
        div_term = torch.exp(
            torch.arange(0, args.n_embd, 2) * -(math.log(10000.0) / args.n_embd)
        )
        # 分别计算 sin、cos 结果
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # 在最前面插入一个 batch 维度，pe 原来的形状是 (block_size, n_embd)，变成(1, block_size, n_embd)，多了一个 batch 维（大小为 1），与输入 x 的形状对齐。
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 将位置编码加到 Embedding 结果上
        show("PositionalEncoding 输入 x", x)
        show("预计算的位置编码 pe", self.pe)
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        show("加上位置编码后 x", x)
        return x


In [ ]:
# 演示：位置编码不改变输入维度
pos_enc = PositionalEncoding(args)
x = torch.randn(2, 8, args.n_embd)
show("位置编码演示输入", x)
pos_out = pos_enc(x)
show("位置编码演示输出", pos_out)


### 2.3.3 一个完整的 Transformer

经过 tokenizer 映射后的输出先经过 Embedding 层和 Positional Encoding 层编码，然后进入 N 个 Encoder 和 N 个 Decoder，最后经过一个线性层（和 Softmax）得到输出。正文实现采用 Pre-Norm 结构，即 LayerNorm 放在 Attention 层之前。

前向过程中打印了每一步的维度：`idx (B, T)` → `tok_emb (B, T, n_embd)` → `enc_out (B, T, n_embd)` → `decoder out (B, T, n_embd)` → `logits (B, T, vocab_size)`。

In [35]:
class Transformer(nn.Module):
    '''整体模型'''
    def __init__(self, args):
        super().__init__()
        # 必须输入词表大小和 block size
        assert args.vocab_size is not None
        assert args.block_size is not None
        self.args = args
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(args.vocab_size, args.n_embd),
            wpe = PositionalEncoding(args),
            drop = nn.Dropout(args.dropout),
            encoder = Encoder(args),
            decoder = Decoder(args),
        ))
        # 最后的线性层，输入是 n_embd，输出是词表大小
        self.lm_head = nn.Linear(args.n_embd, args.vocab_size, bias=False)

        # 初始化所有的权重
        self.apply(self._init_weights)

        # 查看所有参数的数量
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    '''统计所有参数的数量'''
    def get_num_params(self, non_embedding=False):
        # non_embedding: 是否统计 embedding 的参数
        n_params = sum(p.numel() for p in self.parameters())
        # 如果不统计 embedding 的参数，就减去
        if non_embedding:
            n_params -= self.transformer.wte.weight.numel()
        return n_params

    '''初始化权重'''
    def _init_weights(self, module):
        # 线性层和 Embedding 层初始化为正则分布
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    '''前向计算函数'''
    def forward(self, idx, targets=None):
        # 输入为 idx，维度为 (batch size, sequence length, 1)；targets 为目标序列，用于计算 loss
        device = idx.device
        b, t = idx.size()
        assert t <= self.args.block_size, f"不能计算该序列，该序列长度为 {t}, 最大序列长度只有 {self.args.block_size}"
        print("=" * 70)
        show("Transformer 输入 idx", idx)

        # 通过 self.transformer
        # 首先将输入 idx 通过 Embedding 层，得到维度为 (batch size, sequence length, n_embd)
        # 通过 Embedding 层
        tok_emb = self.transformer.wte(idx)
        show("Embedding 后 tok_emb", tok_emb)
        # 然后通过位置编码
        pos_emb = self.transformer.wpe(tok_emb)
        # 再进行 Dropout
        x = self.transformer.drop(pos_emb)
        show("位置编码 + Dropout 后 x", x)
        # 然后通过 Encoder
        enc_out = self.transformer.encoder(x)
        show("Encoder 输出 enc_out", enc_out)
        # 再通过 Decoder
        x = self.transformer.decoder(x, enc_out)
        show("Decoder 输出 x", x)

        if targets is not None:
            # 训练阶段，如果我们给了 targets，就计算 loss
            # 先通过最后的 Linear 层，得到维度为 (batch size, sequence length, vocab size)
            logits = self.lm_head(x)
            show("lm_head 输出 logits", logits)
            show("targets", targets)
            # 再跟 targets 计算交叉熵
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            print(f"loss: {loss.item():.4f}")
        else:
            # 推理阶段，我们只需要 logits，loss 为 None
            # 取 -1 是只取序列中的最后一个作为输出
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            show("推理模式 logits（仅最后一个时间步）", logits)
            loss = None

        return logits, loss


## 逐步拆解 `x[:, [-1], :]`

### 一、先看形状
假设 batch_size = 2，序列长度 = 5，嵌入维度 n_embd = 64，词表大小 = 1000。
那么推理阶段 `x` 的形状是：
```
x: (2, 5, 64)
   ↑  ↑  ↑
  batch 序列长度 嵌入维度
```
`x` 里存的是：**每个 batch、序列中每个位置**经过 Encoder/Decoder 后的输出向量。每个位置都有一个 64 维向量，这个向量“浓缩了”该位置的信息（包括前文语境）。
### 二、`x[:, [-1], :]` 到底取了什么？
Python 索引语法是 `x[第0维, 第1维, 第2维]`，其中 `:` 表示“全部保留”。
```python
x[:, [-1], :]
#  ↓    ↓    ↓
#  |    |    └── 第2维（嵌入维度）：全部保留，64 维都要
#  |    └─────── 第1维（序列位置）：只取最后一个位置！
#  └──────────── 第0维（batch）：全部保留，2 个样本都要
```
- `-1` 在 Python 里表示“最后一个”。
- 所以 `x[:, [-1], :]` = **对每个 batch 样本，只取序列最后一个位置的那个 64 维向量**。
结果形状：
```
x[:, [-1], :]: (2, 1, 64)
                  ↑
              序列长度变成 1（只剩最后一个位置）
```
对比一下，如果写 `x[:, -1, :]`（不带方括号）：
```
x[:, -1, :]: (2, 64)
```
序列那一维直接被“挤掉”了，从 3 维变 2 维。这就是为什么代码注释里说用列表 `[-1]` 是为了 **preserve the time dim（保留时间维）**——保持 3 维 `(batch, seq_len, dim)` 结构，这样后续 lm_head 的输出也能保持统一的 3 维格式。
### 三、为什么只取最后一个位置？
这是**自回归语言模型**的核心逻辑：**预测下一个词，只看最后一个位置就够了**。
举个具体例子，假设输入序列是：
```
["我", "爱", "北京", "天安", "门"]   ← 5 个 token
```
Transformer 处理完后，第 5 个位置（“门”对应的位置）的输出向量，已经通过注意力机制**吸收了前面所有词的信息**（“我”、“爱”、“北京”、“天安”、“门”全部看过了）。
所以：
- 想知道“下一个词是什么”→ 只需要问第 5 个位置 → 取 `x[:, [-1], :]`
- 前面 4 个位置的输出在推理时**完全用不上**，算了也是浪费
### 四、过 lm_head 之后
```python
logits = self.lm_head(x[:, [-1], :])
```
`lm_head` 是一个线性层：`64 维 → 1000 维`（n_embd → vocab_size），作用是把每个位置的向量“翻译”成对词表中每个词的打分。
```
x[:, [-1], :]      (2, 1, 64)
        │
        ▼  lm_head: 64 → 1000
logits             (2, 1, 1000)
```
这 1000 个数字就是：对每个 batch 样本，**词表中每个词作为“下一个词”的得分**。
后续就可以：
```python
probs = F.softmax(logits[:, -1, :], dim=-1)   # 转成概率
next_token = torch.multinomial(probs, 1)       # 采样出下一个词
# 把 next_token 拼到输入序列末尾，重复整个过程 → 逐词生成文本
```
### 五、和训练阶段对比
| | 训练 `lm_head(x)` | 推理 `lm_head(x[:, [-1], :])` |
|---|---|---|
| 输入形状 | `(2, 5, 64)` 全部 5 个位置 | `(2, 1, 64)` 只有最后 1 个位置 |
| 输出形状 | `(2, 5, 1000)` | `(2, 1, 1000)` |
| 含义 | 序列中每个位置的下一词预测 | 只有下一个词的预测 |
| 用途 | 5 个位置都要和 targets 对齐算 loss | 采样一个词，拼回去，循环生成 |
**一句话总结**：训练时需要每个位置都预测（才能算总 loss），推理时只需要最后一个位置的预测（因为每次只生成一个词），`x[:, [-1], :]` 就是“从输出中挑出最后一个位置，且保持维度结构不变”的操作。

## 详解 `F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)`
这行代码计算的是语言模型的训练损失。我们逐步拆解。
### 一、为什么需要 `.view()` 变形？
#### 训练阶段的数据形状
训练时，logits 和 targets 的原始形状是：
```
logits: (batch_size, seq_len, vocab_size)   # 例如 (2, 5, 1000)
targets: (batch_size, seq_len)              # 例如 (2, 5)
```
logits 中每个位置都有对词表 1000 个词的预测分数；targets 中每个位置是对应的**正确答案**（词表中词的 ID）。
#### 问题所在
`F.cross_entropy` 要求输入形状为：
- 预测：`(N, C)` —— N 个样本，每个样本 C 维分数
- 标签：`(N,)` —— N 个整数标签
它不接受 3 维输入，所以必须把 `(2, 5, 1000)` 拍平成 `(10, 1000)`。
#### `.view(-1, ...)` 做了什么
```python
logits.view(-1, logits.size(-1))
#    ↑              ↑
#  自动计算      保留最后一维 = vocab_size
```
- `logits.size(-1)` = 1000（vocab_size），最后一维保持不变
- `-1` 表示"剩余维度自动压平"
变形结果：
```
(2, 5, 1000)  ──view──►  (10, 1000)
                          ↑
              2 × 5 = 10 个位置，每个位置 1000 维分数
```
同样：
```
targets: (2, 5)  ──view(-1)──►  (10,)
```
相当于把整个 batch 的所有时间步**摊平成一行行的独立样本**：
```
第 0 行 = batch 0 的位置 0 的预测 vs 标签
第 1 行 = batch 0 的位置 1 的预测 vs 标签
...
第 9 行 = batch 1 的位置 4 的预测 vs 标签
```
交叉熵会**独立地对每个位置计算损失**。
### 二、交叉熵在做什么？
对摊平后的每一个位置（每一行）：
#### 1. logits → 概率分布
先对 1000 个分数做 softmax：
```
logits[i] = [3.2, -1.5, 0.8, ..., 2.1]   # 1000 个原始分数
    │ softmax
    ▼
probs[i]  = [0.15, 0.001, 0.03, ..., 0.09]  # 总和为 1 的概率
```
#### 2. 取出正确答案的概率
假设 `targets[i] = 42`（正确答案的词 ID），则：
```
p_correct = probs[i][42]   # 模型认为正确词的概率
```
#### 3. 计算单个位置的损失
交叉熵公式（就是取负对数）：
```
loss_i = -log(p_correct)
```
直觉理解：
| 模型对正确词的置信度 p | 损失 -log(p) |
|---|---|
| 0.99（很确定，且猜对了） | ≈ 0.01，损失很小 ✅ |
| 0.5（不太确定） | ≈ 0.69 |
| 0.01（很确定，但猜错了） | ≈ 4.6，损失很大 ❌ |
**模型给正确词的概率越高，损失越小**——这就是训练的优化目标。
#### 4. 求平均
最后对 N = 10 个位置的损失取平均，得到一个标量：
```
loss = (loss_0 + loss_1 + ... + loss_9) / 10
```
### 三、`ignore_index=-1` 的作用
```python
F.cross_entropy(..., ignore_index=-1)
```
含义：**标签值为 -1 的位置不参与损失计算**。
#### 为什么需要它？
在 batch 中，不同序列长度可能不同，需要用 padding（填充）补齐到相同长度。例如：
```
序列 A（真实长度 5）:  ["我", "爱", "北京", "天安", "门"]
序列 B（真实长度 3）:  ["你", "好", "吗",   <pad>,  <pad>]
```
padding 部分不是真实内容，不应该被学习。常见做法是把它们的标签设为 -1：
```
targets = [
  [ 152,  458,  302,  881,  993],   # 序列 A，全部有效
  [ 764,  118,  329,   -1,   -1],   # 序列 B，padding 位置为 -1
]
```
计算损失时：
```
序列 A 的 5 个位置 → 正常计算损失
序列 B 的 3 个有效位置 → 正常计算损失
序列 B 的 2 个 padding 位置（-1）→ 跳过 ❌
```
且求平均时分母也只计入有效位置数（8 而不是 10），避免 padding 位置稀释损失。
#### 注意区分：ignore_index vs 特殊 pad token
这里用 -1 而不是 0，是因为 0 通常已经被用作真正的 `<pad>` 词 ID 留给词表了。用 **-1 作为"无效标记"** 可以避免和合法词 ID 冲突——因为任何真实词的 ID 都 ≥ 0，-1 永远不可能是一个真实标签。
### 四、完整流程图示
```
logits (2, 5, 1000) ──view──► (10, 1000) ─┐
                                          ├─► F.cross_entropy ─► 标量 loss
targets (2, 5)      ──view──► (10,)     ──┘
                                          │
                     ignore_index=-1 ────┘
                     （标签为 -1 的行被跳过）
```
### 五、一句话总结
这行代码做的事情是：
1. **摊平**：把 `(batch, seq_len, ...)` 的三维/二维结构压成一行行的独立预测-标签对；
2. **逐位计算交叉熵**：对每个位置，模型给正确词的概率越高，损失越小（`-log p`）；
3. **忽略无效位置**：标签为 -1 的位置（通常是 padding）不参与计算，保证模型只从真实内容中学习；
4. **输出标量**：对所有有效位置取平均，得到用于反向传播的最终 loss。

用演示参数实例化模型并做一次前向计算（token index 为随机生成，仅用于验证结构与张量维度）：

In [36]:
demo_args = ModelArgs(
    n_embd=32,
    n_heads=4,
    dim=32,
    dropout=0.1,
    max_seq_len=64,
    vocab_size=100,
    block_size=64,
    n_layer=2,
)
model = Transformer(demo_args)

# 训练模式：传入 targets，输出 logits (B, T, vocab_size) 并计算 loss
idx = torch.randint(0, demo_args.vocab_size, (1, 8))
targets = torch.randint(0, demo_args.vocab_size, (1, 8))
logits, loss = model(idx, targets)
print("logits:", tuple(logits.shape), "loss:", round(loss.item(), 4))

# 推理模式：不传 targets，仅输出最后一个时间步的 logits (B, 1, vocab_size)
logits, loss = model(idx)
print("inference logits:", tuple(logits.shape), "loss:", loss)


number of parameters: 0.04M
Transformer 输入 idx: (1, 8)
Embedding 后 tok_emb: (1, 8, 32)
PositionalEncoding 输入 x: (1, 8, 32)
预计算的位置编码 pe: (1, 64, 32)
加上位置编码后 x: (1, 8, 32)
位置编码 + Dropout 后 x: (1, 8, 32)
--- Encoder Layer 0 ---
EncoderLayer 输入 x: (1, 8, 32)
LayerNorm 输入 x: (1, 8, 32)
按最后一维求均值 mean: (1, 8, 1)
按最后一维求标准差 std: (1, 8, 1)
LayerNorm 输出 out: (1, 8, 32)
MultiHeadAttention 输入 q: (1, 8, 32)
MultiHeadAttention 输入 k: (1, 8, 32)
MultiHeadAttention 输入 v: (1, 8, 32)
线性投影后 xq: (1, 8, 32)
线性投影后 xk: (1, 8, 32)
线性投影后 xv: (1, 8, 32)
拆分为多头后 xq（view）: (1, 8, 4, 8)
交换维度后 xq（transpose）: (1, 4, 8, 8)
头数 n_heads: 4, 每个头维度 head_dim: 8
QK^T / sqrt(d_k) -> scores: (1, 4, 8, 8)
softmax 后 scores: (1, 4, 8, 8)
scores @ xv -> output: (1, 4, 8, 8)
合并多头后 output: (1, 8, 32)
输出投影后 output: (1, 8, 32)
掩码/自注意力 + 残差后 h: (1, 8, 32)
LayerNorm 输入 x: (1, 8, 32)
按最后一维求均值 mean: (1, 8, 1)
按最后一维求标准差 std: (1, 8, 1)
LayerNorm 输出 out: (1, 8, 32)
MLP 输入 x: (1, 8, 32)
w1 线性变换后: (1, 8, 32)
ReLU 激活后: (1, 8, 32)
w2 + dropout 后: 

### 使用 Forward 包装自动采集维度变化

上面的 `show()` 打印需要写在模型代码里。另一种做法是不改动模型定义：在运行前临时包装每个子模块的 `forward`，前向一次后自动汇总各模块的输入/输出维度。

> 注意：书中部分代码直接调用 `.forward()`（如 `self.attention.forward(...)`），而 `register_forward_hook` 只在通过 `__call__` 调用时触发，会漏掉这些模块，因此这里直接包装 `forward` 方法。

- 优点：不需要修改源码，能自动覆盖 Encoder/Decoder 的每一层；
- 局限：只能记录模块边界，`view`、`transpose` 等模块内部的张量仍需依靠上面的打印。

下表按前向执行顺序列出关键模块（已过滤 `Linear`、`Dropout` 等细节模块）：

In [37]:
import contextlib
import functools
import io

import pandas as pd

# 记录每个模块的输入输出维度
records = []


def _shape(obj):
    '''递归获取张量形状，兼容模块返回 tuple 的情况'''
    if torch.is_tensor(obj):
        return tuple(obj.shape)
    if isinstance(obj, (tuple, list)):
        return "(" + ", ".join(str(_shape(o)) for o in obj) + ")"
    return type(obj).__name__


def make_wrapped(name, module, original_forward):
    '''包装 forward：执行原逻辑后记录输入输出'''
    @functools.wraps(original_forward)
    def wrapped(*args, **kwargs):
        output = original_forward(*args, **kwargs)
        records.append({
            "name": name,
            "type": type(module).__name__,
            "input": [_shape(a) for a in args if torch.is_tensor(a)],
            "output": _shape(output),
        })
        return output
    return wrapped


# 临时替换每个子模块的 forward（包括根模块）
records.clear()
patched = []
for name, module in model.named_modules():
    original_forward = module.forward
    module.forward = make_wrapped(name, module, original_forward)
    patched.append(module)

# 执行一次前向，包装后的 forward 会自动记录
model.eval()
with torch.no_grad(), contextlib.redirect_stdout(io.StringIO()):
    model(idx)

# 恢复原始 forward
for module in patched:
    del module.forward

# 只保留关键模块，避免 Linear/Dropout 等细节淹没维度流转
keep_types = {
    "Transformer", "PositionalEncoding", "Embedding",
    "Encoder", "EncoderLayer", "Decoder", "DecoderLayer",
    "MultiHeadAttention", "MLP", "LayerNorm",
}
rows = [
    {
        "模块": r["name"],
        "类型": r["type"],
        "输入维度": " | ".join(str(s) for s in r["input"]),
        "输出维度": str(r["output"]),
    }
    for r in records if r["type"] in keep_types
]

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

,模块,类型,输入维度,输出维度
0,transformer.wte,Embedding,"(1, 8)","(1, 8, 32)"
1,transformer.wpe,PositionalEncoding,"(1, 8, 32)","(1, 8, 32)"
2,transformer.encoder.layers.0.attention_norm,LayerNorm,"(1, 8, 32)","(1, 8, 32)"
3,transformer.encoder.layers.0.attention,MultiHeadAttention,"(1, 8, 32) | (1, 8, 32) | (1, 8, 32)","(1, 8, 32)"
4,transformer.encoder.layers.0.fnn_norm,LayerNorm,"(1, 8, 32)","(1, 8, 32)"
5,transformer.encoder.layers.0.feed_forward,MLP,"(1, 8, 32)","(1, 8, 32)"
6,transformer.encoder.layers.0,EncoderLayer,"(1, 8, 32)","(1, 8, 32)"
7,transformer.encoder.layers.1.attention_norm,LayerNorm,"(1, 8, 32)","(1, 8, 32)"
8,transformer.encoder.layers.1.attention,MultiHeadAttention,"(1, 8, 32) | (1, 8, 32) | (1, 8, 32)","(1, 8, 32)"
9,transformer.encoder.layers.1.fnn_norm,LayerNorm,"(1, 8, 32)","(1, 8, 32)"


#### 维度流转图

下面把上表的维度数据画成图：上方是主流程（token idx → Embedding → 位置编码 → Encoder → Decoder → lm_head），下方按“输入表示 / Encoder / Decoder / 整体”分组展示每个模块的输入、输出与维度变化，每个维度用宽度成比例的小色块表示。

In [38]:
import ast
import html as html_lib
import re

from IPython.display import HTML, display

# 模块类型 -> (显示名, 主色, 背景色)
TYPE_META = {
    "Embedding": ("Embedding", "#2563eb", "#eff6ff"),
    "PositionalEncoding": ("位置编码", "#7c3aed", "#f5f3ff"),
    "LayerNorm": ("LayerNorm", "#64748b", "#f8fafc"),
    "MultiHeadAttention": ("多头注意力", "#ea580c", "#fff7ed"),
    "MLP": ("前馈网络", "#16a34a", "#f0fdf4"),
    "EncoderLayer": ("EncoderLayer", "#0891b2", "#ecfeff"),
    "Encoder": ("Encoder", "#0e7490", "#ecfeff"),
    "DecoderLayer": ("DecoderLayer", "#db2777", "#fdf2f8"),
    "Decoder": ("Decoder", "#be185d", "#fdf2f8"),
    "Transformer": ("整体模型", "#111827", "#f3f4f6"),
}


def parse_dims(text):
    '''从 "(1, 8, 32)" 这样的字符串中解析维度'''
    match = re.search(r"\(\s*\d+(?:\s*,\s*\d+)*\s*\)", str(text))
    if not match:
        return []
    try:
        return [int(v) for v in ast.literal_eval(match.group(0))]
    except Exception:
        return []


def clean_shape(text):
    '''只保留第一个形状，便于展示 tuple 输出'''
    dims = parse_dims(text)
    return "(" + ", ".join(str(d) for d in dims) + ")" if dims else str(text)


def short_name(name):
    return name.split(".")[-1] if name else "Transformer"


def group_of(record):
    '''按模块名把组件分到四个阶段'''
    if record["type"] == "Transformer":
        return "整体"
    if ".encoder" in record["name"] or record["type"] in ("Encoder", "EncoderLayer"):
        return "Encoder"
    if ".decoder" in record["name"] or record["type"] in ("Decoder", "DecoderLayer"):
        return "Decoder"
    return "输入表示"


def input_text(inputs):
    '''相同输入合并显示，如 (1, 8, 32) ×3'''
    merged = []
    for value in inputs:
        shape = str(value)
        if merged and merged[-1][0] == shape:
            merged[-1][1] += 1
        else:
            merged.append([shape, 1])
    return " + ".join(s if n == 1 else f"{s} ×{n}" for s, n in merged)


def shape_bar(dims):
    '''把维度画成宽度成比例的小色块'''
    if not dims:
        return ""
    largest = max(dims)
    segments = "".join(
        f'<span class="seg" style="width:{max(18, round(54 * (d / largest) ** 0.5))}px" '
        f'title="维度 {d}">{d}</span>'
        for d in dims
    )
    return f'<div class="bar">{segments}</div>'


def unchanged(record):
    return bool(record["input"]) and all(s == record["output"] for s in record["input"])


def first_record(type_name):
    return next(r for r in records if r["type"] == type_name)


def pipeline_box(label, shape, color):
    return (f'<div class="pbox" style="border-color:{color}">'
            f'<span class="plabel">{label}</span>'
            f'<span class="pshape">{shape}</span></div>')


# 主流程：从 records 中取出各阶段输出维度
root = first_record("Transformer")
embedding = first_record("Embedding")
pos_encoding = first_record("PositionalEncoding")
encoder = first_record("Encoder")
decoder = first_record("Decoder")

pipeline = "".join([
    pipeline_box("token idx", root["input"][0], "#94a3b8"),
    '<span class="parrow">→</span>',
    pipeline_box("Embedding", embedding["output"], "#2563eb"),
    '<span class="parrow">→</span>',
    pipeline_box("+ 位置编码", pos_encoding["output"], "#7c3aed"),
    '<span class="parrow">→</span>',
    pipeline_box("Encoder × n_layer", encoder["output"], "#0e7490"),
    '<span class="parrow">→</span>',
    pipeline_box("Decoder × n_layer", decoder["output"], "#be185d"),
    '<span class="parrow">→</span>',
    pipeline_box("lm_head", clean_shape(root["output"]), "#111827"),
])

# 按阶段生成模块卡片
sections = ["输入表示", "Encoder", "Decoder", "整体"]
cards_html = []
for section in sections:
    items = [r for r in records if r["type"] in keep_types and group_of(r) == section]
    if not items:
        continue
    cards = []
    for r in items:
        label, color, background = TYPE_META.get(r["type"], (r["type"], "#64748b", "#f8fafc"))
        badge = ('<span class="badge same">维度不变</span>' if unchanged(r)
                 else '<span class="badge change">维度变化</span>')
        cards.append(
            f'<div class="card" style="border-left-color:{color};background:{background}">'
            f'<div class="card-head">'
            f'<span class="chip" style="color:{color};border-color:{color}">{label}</span>'
            f'<span class="mname">{html_lib.escape(short_name(r["name"]))}</span>{badge}</div>'
            f'<div class="io">'
            f'<div class="io-row"><span class="tag">输入</span>{html_lib.escape(input_text(r["input"]))}</div>'
            f'<div class="io-row"><span class="tag">输出</span>{html_lib.escape(str(r["output"]))}</div>'
            f'</div>{shape_bar(parse_dims(r["output"]))}</div>'
        )
    cards_html.append(
        f'<div class="section"><div class="section-title">{section}'
        f'<span class="count">{len(items)} 个模块</span></div>'
        f'<div class="grid">{"".join(cards)}</div></div>'
    )

STYLE = """
<style>
.dimchart { font-family: "Segoe UI", "Microsoft YaHei", system-ui, sans-serif; color: #0f172a; margin-top: 4px; }
.dimchart h4 { margin: 0 0 8px; font-size: 15px; }
.pipe { display: flex; flex-wrap: wrap; align-items: center; gap: 6px; padding: 10px 12px; background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 10px; }
.pbox { display: flex; flex-direction: column; gap: 2px; padding: 6px 10px; background: #fff; border: 1px solid; border-radius: 8px; min-width: 96px; }
.plabel { font-size: 11px; color: #475569; }
.pshape { font-family: Consolas, monospace; font-size: 12px; font-weight: 600; }
.parrow { color: #94a3b8; font-size: 14px; }
.section { margin-top: 16px; }
.section-title { font-size: 13px; font-weight: 600; color: #334155; margin-bottom: 8px; }
.count { margin-left: 8px; font-weight: 400; color: #94a3b8; font-size: 11px; }
.grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(300px, 1fr)); gap: 10px; }
.card { border-left: 4px solid; border-radius: 8px; padding: 9px 12px; box-shadow: 0 1px 2px rgba(15, 23, 42, .06); }
.card-head { display: flex; align-items: center; gap: 6px; flex-wrap: wrap; }
.chip { font-size: 10px; padding: 1px 6px; border: 1px solid; border-radius: 999px; }
.mname { font-family: Consolas, monospace; font-size: 11px; color: #334155; }
.badge { margin-left: auto; font-size: 10px; padding: 1px 6px; border-radius: 999px; }
.same { background: #e2e8f0; color: #475569; }
.change { background: #fee2e2; color: #b91c1c; }
.io { margin-top: 6px; display: flex; flex-direction: column; gap: 2px; }
.io-row { font-family: Consolas, monospace; font-size: 11.5px; }
.tag { display: inline-block; width: 30px; color: #94a3b8; font-family: "Segoe UI", "Microsoft YaHei", sans-serif; }
.bar { display: flex; gap: 3px; margin-top: 6px; }
.seg { display: inline-flex; align-items: center; justify-content: center; height: 16px; border-radius: 4px; background: rgba(15, 23, 42, .08); font-size: 9px; color: #334155; }
</style>
"""

fig_html = STYLE + '<div class="dimchart">' + '<h4>Transformer 维度流转图</h4>' + \
    '<div class="pipe">' + pipeline + '</div>' + "".join(cards_html) + '</div>'

display(HTML(fig_html))

### 各组件维度流转汇总

| 组件 | 输入维度 | 输出维度 | 关键中间维度 |
| --- | --- | --- | --- |
| `Embedding` | `(B, T)` | `(B, T, n_embd)` | 词表 `(vocab_size, n_embd)` |
| `PositionalEncoding` | `(B, T, n_embd)` | `(B, T, n_embd)` | 位置编码 `(1, block_size, n_embd)` |
| `LayerNorm` | `(B, T, n_embd)` | `(B, T, n_embd)` | 均值/方差 `(B, T, 1)` |
| `attention` | `(B, T, d_k)` 三份 | `(B, T, d_v)` | scores / 权重 `(B, T, T)` |
| `MultiHeadAttention` | `(B, T, n_embd)` | `(B, T, dim)` | 拆头 `(B, n_heads, T, head_dim)`，scores `(B, n_heads, T, T)` |
| `MLP` | `(B, T, dim)` | `(B, T, dim)` | 隐藏层 `(B, T, hidden_dim)` |
| `Encoder` | `(B, T, n_embd)` | `(B, T, n_embd)` | N × `EncoderLayer` |
| `Decoder` | `(B, T_tgt, n_embd)`，`enc_out (B, T_src, dim)` | `(B, T_tgt, n_embd)` | 交叉注意力 scores `(B, n_heads, T_tgt, T_src)` |
| `lm_head` | `(B, T, n_embd)` | `(B, T, vocab_size)` | 推理时 `T = 1` |

---

除第 0 节的配置类、`show()` 辅助函数与少量运行示例外，以上代码逻辑来自第二章正文，完整脚本可参考 [code/transformer.py](./code/transformer.py)。正文未展开的 Transformer 训练流程不在本章范围内。